# Rice Trait Network and Yield Driver Analysis Pipeline

Author: Jarvis  
Year: 2026

This notebook implements a full computational pipeline for analyzing genotype–trait relationships in rice breeding experiments.

The workflow integrates:

- BLUP mixed model estimation
- Variance components and heritability
- Partial correlation trait networks
- Trait interaction networks
- Machine learning yield predictors
- Random Forest and SHAP explainable AI
- MGIDI multi-trait selection index
- Publication-ready figures


## Workflow Overview

Field experiment data
↓
BLUP mixed model estimation
↓
Variance components
↓
Partial correlation network
↓
Trait interaction network
↓
Machine learning yield predictors
↓
MGIDI multi-trait selection
↓
Elite genotype identification


## Step 1 — Install Required Libraries

In [ ]:
!pip install pandas numpy matplotlib seaborn networkx scikit-learn statsmodels shap openpyxl

## Step 2 — Import Libraries

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.covariance import GraphicalLassoCV
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import r2_score
from sklearn.decomposition import FactorAnalysis

import shap
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')

## Step 3 — Upload Dataset

Upload the dataset containing genotype and trait measurements.

Expected dataset structure:

| Genotype | Replication | Trait1 | Trait2 | Trait3 |
|----------|------------|--------|--------|--------|


In [ ]:
from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

if file_name.endswith('.xlsx'):
    df = pd.read_excel(file_name)
else:
    df = pd.read_csv(file_name)

df.columns = df.columns.str.strip()

print('Dataset shape:', df.shape)
df.head()

## Step 4 — Identify Traits

In [ ]:
GENOTYPE_COL = 'Genotype'
REPLICATION_COL = 'Replication'

traits = [c for c in df.columns if c not in [GENOTYPE_COL, REPLICATION_COL]]

genotypes = df[GENOTYPE_COL].unique()
replications = df[REPLICATION_COL].unique()

print('Traits detected:', traits)

## Step 5 — BLUP Estimation

In [ ]:
blup = pd.DataFrame(index=genotypes, columns=traits)

for trait in traits:
    formula = f'Q("{trait}") ~ C({REPLICATION_COL})'
    md = smf.mixedlm(formula, df, groups=df[GENOTYPE_COL])
    m = md.fit()

    mu = m.fe_params[0]

    for g in genotypes:
        u = float(np.asarray(m.random_effects.get(g, [0]))[0])
        blup.loc[g, trait] = mu + u

blup.head()

## Step 6 — Standardize BLUP Matrix

In [ ]:
scaler = StandardScaler()

Z = pd.DataFrame(
    scaler.fit_transform(blup),
    columns=blup.columns,
    index=blup.index
)

## Step 7 — Partial Correlation Network

In [ ]:
gl = GraphicalLassoCV()
gl.fit(Z.values)

P = gl.precision_

d = np.sqrt(np.diag(P))

pcorr = -P / np.outer(d, d)

np.fill_diagonal(pcorr, 1)

pcorr_df = pd.DataFrame(pcorr, index=traits, columns=traits)

## Step 8 — Partial Correlation Heatmap

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(pcorr_df, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('BLUP-based Partial Correlation Heatmap')
plt.show()

## Step 9 — Trait Interaction Network

In [ ]:
G = nx.Graph()

for i in range(len(traits)):
    for j in range(i+1,len(traits)):
        r = pcorr_df.iloc[i,j]
        if abs(r) >= 0.2:
            G.add_edge(traits[i], traits[j], weight=r)

pos = nx.spring_layout(G, seed=42)

plt.figure(figsize=(10,8))
nx.draw(G, pos, with_labels=True, node_size=2000)
plt.title('Trait Interaction Network')
plt.show()

## Step 10 — Yield Predictor Model

In [ ]:
YIELD_TRAIT = 'Grain yield per hill'

X = Z.drop(columns=[YIELD_TRAIT])
y = Z[YIELD_TRAIT]

model = LinearRegression()
model.fit(X,y)

importance = pd.Series(np.abs(model.coef_), index=X.columns)
importance.sort_values(ascending=False)

## Step 11 — Random Forest Feature Importance

In [ ]:
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X,y)

rf_importance = pd.Series(rf.feature_importances_, index=X.columns)
rf_importance.sort_values(ascending=False)

## Step 12 — SHAP Explainable AI

In [ ]:
gb = GradientBoostingRegressor()
gb.fit(X,y)

explainer = shap.Explainer(gb,X)
shap_values = explainer(X)

shap.plots.bar(shap_values)

## Step 13 — MGIDI Multi-Trait Selection

In [ ]:
gmeans = df.groupby(GENOTYPE_COL)[traits].mean()

Zg = StandardScaler().fit_transform(gmeans)

fa = FactorAnalysis(n_components=min(5,len(traits)))
F = fa.fit_transform(Zg)

ideotype = F.max(axis=0)

mgidi = np.sqrt(((F - ideotype)**2).sum(axis=1))

mgidi = pd.Series(mgidi, index=gmeans.index).sort_values()
mgidi.head()

## Step 14 — Save Outputs

In [ ]:
os.makedirs('outputs', exist_ok=True)

blup.to_csv('outputs/BLUP_matrix.csv')
pcorr_df.to_csv('outputs/partial_correlation_matrix.csv')
importance.to_csv('outputs/yield_predictors.csv')
rf_importance.to_csv('outputs/random_forest_importance.csv')
mgidi.to_csv('outputs/mgidi_scores.csv')